In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

In [ ]:
#df = pd.read_csv('/content/drive/MyDrive/Data Science Personal Projects/book_reviews.csv')
#df.head()

In [ ]:
books = pd.read_csv('/content/drive/MyDrive/Data Science Personal Projects/books.csv')
rate = pd.read_csv('/content/drive/MyDrive/Data Science Personal Projects/ratings.csv')

In [ ]:
rate.head()

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


I am going to start with the most basic form of this model and then expand in complexity to see how it impacts the results. My first step in that process will be simplifying the ratings to binary values; all ratings under 5 will be converted to 0, and all ratings above a 5 will be converted to 1.

In [ ]:
# iterating through the matrix to replace ratings with binary values
# making the assumption that books rated less than 5 are not liked (i.e. 0)

rate.loc[rate['Book-Rating'] < 5, 'Book-Rating'] = 0
rate.loc[rate['Book-Rating'] >= 5, 'Book-Rating'] = 1

## User-Item Matrix

In [ ]:
# dim 1
users = rate['User-ID'].unique()
# dim 2
items = rate['ISBN'].unique()
print('dims:',len(users),len(items))

# start with empty matrix
#A = np.zeros((len(users), len(items)))

# map the user/item values
user_mapper = dict(zip(users, range(len(users)))) #{user_id: i for i, user_id in enumerate(users)}
item_mapper = dict(zip(items, range(len(items)))) #{item_id: j for j, item_id in enumerate(items)}

# map to matrix indicies
user_indices = [user_mapper[user['User-ID']] for i,user in rate.iterrows()]
item_indices = [item_mapper[item['ISBN']] for i,item in rate.iterrows()]

# create the sparse matrix
matrix = csr_matrix((rate['Book-Rating'], (user_indices, item_indices)),
                    shape=(len(users), len(items)))

print(matrix)

# filling in the matrix (dense representation)
#for i, row in rate.iterrows():
#    user_index = user_mapper[row['User-ID']]
#    item_index = item_mapper[row['ISBN']]
#    A[user_index, item_index] = row['Book-Rating']

#print(A)

dims: 104553 335928
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 1143953 stored elements and shape (104553, 335928)>
  Coords	Values
  (0, 0)	0
  (1, 1)	1
  (2, 2)	0
  (3, 3)	0
  (3, 4)	1
  (4, 5)	0
  (5, 6)	1
  (6, 7)	1
  (7, 8)	1
  (8, 9)	0
  (8, 10)	0
  (8, 11)	0
  (8, 12)	0
  (8, 13)	0
  (8, 14)	0
  (9, 15)	1
  (9, 16)	0
  (9, 17)	0
  (9, 18)	1
  (9, 19)	1
  (9, 20)	1
  (9, 21)	0
  (9, 22)	1
  (10, 23)	1
  (10, 24)	0
  :	:
  (104546, 116002)	0
  (104546, 252450)	0
  (104546, 335924)	0
  (104547, 145000)	0
  (104548, 2077)	0
  (104548, 3074)	1
  (104548, 8495)	0
  (104548, 8801)	1
  (104548, 11945)	0
  (104548, 19998)	0
  (104548, 26659)	0
  (104548, 34343)	0
  (104548, 46249)	0
  (104548, 50134)	0
  (104548, 50141)	1
  (104548, 56377)	0
  (104548, 193272)	0
  (104548, 223892)	1
  (104548, 281272)	0
  (104548, 335925)	0
  (104548, 335926)	1
  (104549, 7248)	0
  (104550, 11963)	1
  (104551, 77972)	1
  (104552, 335927)	1


## Binary-Rating Model

In [ ]:
knn = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=3, n_jobs=-1)
knn.fit(matrix)

NearestNeighbors(algorithm='brute', metric='cosine', n_jobs=-1, n_neighbors=3)

In [ ]:
# getting 10 books read and liked by the user (if possible)

# sorting to pick a user that has interacted with 10+ books
rate_sorted = rate.sort_values(['User-ID','Book-Rating'], ascending=[True, False])
# we are going to use User-ID = 8 because they have a large number of reviewed books

# pulling user id 8
filter_user8 = rate_sorted[rate_sorted['User-ID'] == 8]['ISBN'].tolist()

print('Book ISBNs user has read', filter_user8[:10])

Book ISBNs user has read ['0002005018', '074322678X', '0887841740', '1552041778', '1567407781', '1575663937', '1881320189', '0060973129', '0374157065', '0393045218']


In [ ]:
# recommending a similar book for each of the liked books by user8

distances1=[]
indices1=[]
for i in filter_user8:
  indx = item_mapper[i]
  distances, indices = knn.kneighbors(matrix[indx],n_neighbors=3)
  indices = indices.flatten()
  indices = indices[1:]
  indices1.extend(indices)
print("Items to be recommended: ",indices1)

Items to be recommended:  [np.int64(104542), np.int64(104541), np.int64(104542), np.int64(104541), np.int64(12166), np.int64(104543), np.int64(91420), np.int64(74241), np.int64(8931), np.int64(19871), np.int64(75531), np.int64(79067), np.int64(93764), np.int64(82318), np.int64(57512), np.int64(57315), np.int64(102957), np.int64(76540), np.int64(1111), np.int64(93916), np.int64(53381), np.int64(12668), np.int64(64740), np.int64(97026), np.int64(24211), np.int64(104543), np.int64(7779), np.int64(95319), np.int64(94937), np.int64(104543), np.int64(919), np.int64(97145), np.int64(8927), np.int64(41180), np.int64(8930), np.int64(63160)]


In [ ]:
# map indices back to book titles

for i in range(10):
  book_t = books.loc[books['ISBN']==filter_user8[i], 'Book-Title'].iloc[0]
  print('For the book titled:\n', book_t)
  rec_isbn = list(item_mapper.keys())[list(item_mapper.values()).index(item_indices[indices1[i]])]
  rec_book_t = books.loc[books['ISBN']==rec_isbn, 'Book-Title'].iloc[0]
  print('We recommend the book:\n', rec_book_t)

For the book titled:
 Clara Callan
We recommend the book:
 The Woman of Rome : A Novel
For the book titled:
 Where You'll Find Me: And Other Stories
We recommend the book:
 Clean Break (Kate Brannigan Series (4th Book).)
For the book titled:
 The Middle Stories
We recommend the book:
 The Woman of Rome : A Novel
For the book titled:
 Jane Doe
We recommend the book:
 Clean Break (Kate Brannigan Series (4th Book).)
For the book titled:
 The Witchfinder (Amos Walker Mystery Series)
We recommend the book:
 Kartography
For the book titled:
 More Cunning Than Man: A Social History of Rats and Man
We recommend the book:
 Edgar Allan Poe: The Tell-Tale Heart/the Pit and the Pendulum/the Sleeper
For the book titled:
 Goodbye to the Buttermilk Sky
We recommend the book:
 Beneath the Blonde (Mask Noir)
For the book titled:
 Decision in Normandy
We recommend the book:
 Dead Girls Don't Wear Diamonds (Blackbird Sisters Mysteries)
For the book titled:
 Flu: The Story of the Great Influenza Pandemic 